# Ratings: leaderboards and sanity checks (Phase 5)

Checks on the outputs of `scripts/build_ratings.py` (`docs/methodology.md` section 12):

1. Team leaderboards, with 80% intervals.
2. **Low blocks**: do teams that defend deeper concede less space *behind*? (PLAN's sanity check.)
3. Match summaries for a few well-known games.
4. Player leaderboards: runners, carriers, defenders (experimental).
5. **PFF benchmark**: player ratings against PFF analysts' event grades, and the decision gap
   against PFF's "better option" flags.
6. **V5**: team-match ratings against results.

Ratings describe space and progression, not danger (V3 decision, `docs/validation.md`).

In [1]:
import pandas as pd
from scipy import stats

from xspace.config import ACTIONS_DIR, PLAYERS_DIR, RATINGS_DIR, TIMELINE_DIR
from xspace.metrics.timeline import read_timeline
from xspace.validation.pff_grades import better_options, player_grades

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
tm = pd.read_parquet(RATINGS_DIR / "team_match.parquet")
tp = pd.read_parquet(RATINGS_DIR / "team_profile.parquet")
pl = pd.read_parquet(RATINGS_DIR / "player.parquet")
lm = pd.read_parquet(RATINGS_DIR / "leak_map.parquet")
wc = tp[tp["source"] == "pff"]
print(len(tm), "team-matches,", len(tp), "teams,", len(pl), "players")

142 team-matches, 42 teams, 843 players


## 1. Team leaderboards (World Cup)

In [2]:
cols = ["team", "matches", "xspace_conceded", "xspace_conceded_lo", "xspace_conceded_hi",
        "xspace_conceded_behind", "leaks_most", "grade_defence_space"]
print("Concede least / most space (xT·m² per second of the opponent's possession)")
print(wc.sort_values("xspace_conceded")[cols].round(2).to_string(index=False))

Concede least / most space (xT·m² per second of the opponent's possession)
         team  matches  xspace_conceded  xspace_conceded_lo  xspace_conceded_hi  xspace_conceded_behind leaks_most  grade_defence_space
        Spain        4             4.30                4.11                4.47                    2.94     behind                96.43
        Qatar        3             4.37                4.13                4.61                    2.90    between                94.05
       Mexico        3             4.61                4.37                4.84                    2.93   in_front                89.29
United States        4             4.79                4.57                4.98                    3.05       wide                86.90
      Denmark        3             4.81                4.60                5.01                    3.21   in_front                84.52
      Belgium        3             4.89                4.62                5.13                    3.01    be

In [3]:
cols = ["team", "xspace_created", "xspace_created_lo", "xspace_created_hi", "exploit_rate",
        "exploit_rate_shrunk", "regain_exploit_share", "grade_attack_space", "grade_attack_use"]
top = wc.sort_values("xspace_created", ascending=False)[cols]
print(top.round(4).head(12).to_string(index=False))

         team  xspace_created  xspace_created_lo  xspace_created_hi  exploit_rate  exploit_rate_shrunk  regain_exploit_share  grade_attack_space  grade_attack_use
       Brazil          6.7437             6.6004             6.8864        0.0071               0.0076                0.0401             98.8095           32.1429
       Mexico          6.6198             6.3960             6.8509        0.0131               0.0102                0.0230             96.4286           94.0476
      Germany          6.5745             6.3953             6.7465        0.0134               0.0108                0.0578             94.0476           96.4286
      England          6.3527             6.2181             6.4880        0.0061               0.0070                0.0205             91.6667           10.7143
     Cameroon          6.2142             5.9662             6.4421        0.0009               0.0056                0.0000             86.9048            1.1905
       Canada         

## 2. Low blocks and space behind

Line height: the defending back line's distance from its own goal while the opponent has the
ball in the **middle third** (so the comparison isn't driven by where the ball was). A deeper
block should leave less space behind it, and more in front.

In [4]:
rows = []
for f in sorted(TIMELINE_DIR.glob("pff_*.parquet")):
    t, meta = read_timeline(f)
    t = t[(t["status"] == "ok") & (t["third"] == "middle")]
    for side in (0, 1):
        defending = meta["away"] if side == 0 else meta["home"]
        d = t[t["possession_side"] == side]
        rows.append({"team": defending, "n": len(d),
                     "depth_sum": (52.5 - d["back_line"]).sum(),
                     "behind_sum": d["behind"].sum(), "front_sum": d["in_front"].sum()})
lines = pd.DataFrame(rows).groupby("team").sum()
lines["line_depth_m"] = lines["depth_sum"] / lines["n"]
lines["behind_mid"] = lines["behind_sum"] / lines["n"]
lines["in_front_mid"] = lines["front_sum"] / lines["n"]
lines = lines[["line_depth_m", "behind_mid", "in_front_mid"]].sort_values("line_depth_m")
r_b = stats.spearmanr(lines["line_depth_m"], lines["behind_mid"])
r_f = stats.spearmanr(lines["line_depth_m"], lines["in_front_mid"])
print(f"Spearman, line height vs space behind: {r_b.statistic:.2f} (p = {r_b.pvalue:.3f}); "
      f"vs space in front: {r_f.statistic:.2f} (p = {r_f.pvalue:.3f})  [32 teams]")
print("Deepest and highest lines:")
print(pd.concat([lines.head(5), lines.tail(5)]).round(2).to_string())

Spearman, line height vs space behind: -0.42 (p = 0.017); vs space in front: -0.87 (p = 0.000)  [32 teams]
Deepest and highest lines:
              line_depth_m  behind_mid  in_front_mid
team                                                
Costa Rica           31.78        5.04          0.68
Poland               33.05        4.64          0.69
Australia            33.11        4.60          0.66
Morocco              33.12        4.04          0.66
Qatar                33.36        2.68          0.59
Denmark              36.51        3.47          0.48
Canada               37.87        3.69          0.44
Saudi Arabia         38.19        4.20          0.47
Germany              40.02        3.61          0.33
Spain                40.26        2.96          0.28


## 3. Match summaries

In [5]:
show = tm[tm["match_id"].isin(["3816", "10508", "10517", "10514"])]
for _, r in show.iterrows():
    print(f"{r['team']} {r['goals_for']}-{r['goals_against']} {r['opponent']}  "
          f"[space {r['grade_attack_space']:.0f}, use {r['grade_attack_use']:.0f} | "
          f"conceded {r['grade_defence_space']:.0f}, behind {r['grade_defence_behind']:.0f}, "
          f"opp. use {r['grade_defence_use']:.0f}]")
    print("   ", r["summary"])

Morocco 0-0 Spain  [space 8, use 67 | conceded 5, behind 2, opp. use 82]
    Created little space, and used it about as often as most. Conceded a lot of space; opponents used it rarely.
Spain 0-0 Morocco  [space 95, use 18 | conceded 92, behind 88, opp. use 33]
    Created a lot of space, and used it rarely. Conceded little space; opponents used it about as often as most.
Argentina 3-0 Croatia  [space 4, use 33 | conceded 24, behind 29, opp. use 95]
    Created little space, mostly in front of the block, and used it about as often as most. Conceded more than average space, mostly wide of the block; opponents used it rarely.
Croatia 0-3 Argentina  [space 76, use 5 | conceded 96, behind 95, opp. use 67]
    Created more than average space, mostly wide of the block, and used it rarely. Conceded little space, mostly in front of the block; opponents used it about as often as most.
Argentina 3-3 France  [space 5, use 25 | conceded 55, behind 61, opp. use 60]
    Created little space, mostly 

## 4. Player leaderboards

Runners are graded on space held **per 90 minutes in possession** (shrunk), so players of
possession sides aren't rewarded for their team having the ball. Per 90 = per 90 live minutes.

In [6]:
wcp = pl[pl["source"] == "pff"]
r = wcp[wcp["runner"]]
cols = ["name", "team", "minutes", "space_held_per90", "space_held_per90_poss",
        "space_held_per90_poss_shrunk", "receptions_per90_poss_shrunk", "grade_runner_space"]
for g in ["ST", "W", "AM", "CM", "FB"]:
    print(g)
    print(r[r["position_group"] == g].sort_values("space_held_per90_poss_shrunk", ascending=False)
          [cols].head(5).round(1).to_string(index=False))

ST
           name         team  minutes  space_held_per90  space_held_per90_poss  space_held_per90_poss_shrunk  receptions_per90_poss_shrunk  grade_runner_space
    Richarlison       Brazil    219.4             164.9                  308.2                         302.7                           6.1                98.7
  Alvaro Morata        Spain    128.0             206.7                  283.4                         277.9                           6.0                96.2
   Daizen Maeda        Japan    129.3              77.9                  264.6                         254.2                           5.6                93.6
Saleh Al-Shehri Saudi Arabia    123.5             116.6                  258.3                         251.5                           4.7                91.0
 Olivier Giroud       France    283.5             122.2                  250.6                         247.9                           6.7                88.5
W
             name         team  minutes  

In [7]:
c = wcp[wcp["carrier"]]
cols = ["name", "team", "position_group", "n_passes", "exploit_rate", "exploit_rate_shrunk",
        "likely_gap_shrunk", "grade_carrier_use", "grade_carrier_choice"]
top = c.sort_values("exploit_rate_shrunk", ascending=False)[cols]
print(top.head(10).round(4).to_string(index=False))

           name    team position_group  n_passes  exploit_rate  exploit_rate_shrunk  likely_gap_shrunk  grade_carrier_use  grade_carrier_choice
     Leroy Sané Germany              W        58        0.0862               0.0422             0.0142            99.1803                5.7377
  Kylian Mbappé  France              W       207        0.0376               0.0310             0.0133            97.5410                7.3770
 Ali Gholizadeh    Iran              W        63        0.0462               0.0286             0.0091            95.9016               66.3934
 Sofiane Boufal Morocco              W       128        0.0357               0.0280             0.0075            94.2623               86.0656
   Serge Gnabry Germany              W       111        0.0354               0.0268             0.0103            92.6230               41.8033
   Ismaïla Sarr Senegal              W        76        0.0395               0.0268             0.0095            90.9836               

In [8]:
d = wcp[wcp["defender"] & wcp["position_group"].isin(["CB", "FB", "DM"])]
cols = ["name", "team", "position_group", "minutes_out_of_possession", "conceded_per90_def",
        "conceded_per90_def_shrunk", "grade_defender_space"]
for g in ["CB", "FB"]:
    print(g, "(fewest likely options conceded per 90 out of possession)")
    print(d[d["position_group"] == g].sort_values("conceded_per90_def_shrunk")[cols]
          .head(5).round(1).to_string(index=False))

CB (fewest likely options conceded per 90 out of possession)
            name        team position_group  minutes_out_of_possession  conceded_per90_def  conceded_per90_def_shrunk  grade_defender_space
    Abdou Diallo     Senegal             CB                      113.4                73.8                       79.4                  99.4
      Nathan Aké Netherlands             CB                      144.8                83.9                       87.6                  98.1
  Jurrien Timber Netherlands             CB                      119.8                84.1                       88.4                  96.8
 Ibrahima Konaté      France             CB                       94.2                83.1                       88.6                  95.5
Joachim Andersen     Denmark             CB                       76.7                84.5                       90.8                  94.2
FB (fewest likely options conceded per 90 out of possession)
            name    team position_grou

## 5. PFF benchmark

PFF analysts grade every player in every event (0 = as expected, ±0.5 steps). Their grades
mix the choice with its execution, and are judgements, not ground truth: expect positive but
imperfect agreement. Compared per player, within position (partial Spearman: both sides ranked
within the position group), among qualified players.

In [9]:
pg = []
for f in sorted(PLAYERS_DIR.glob("pff_*.parquet")):
    pg.append(player_grades(f.stem.split("_", 1)[1]))
pg = pd.concat(pg).groupby("player_id").sum()
bench = wcp.set_index("player_id").join(pg, how="inner")
p90 = bench["minutes"] / 90
for grp in ("passing", "receiving", "off_ball", "defending"):
    bench[f"pff_{grp}_mean"] = bench[f"{grp}_sum"] / bench[f"{grp}_n"].where(bench[f"{grp}_n"] > 0)
    bench[f"pff_{grp}_per90"] = bench[f"{grp}_sum"] / p90


def within(df, a, b):
    sub = df[[a, b, "position_group"]].dropna()
    ra = sub.groupby("position_group")[a].rank(pct=True)
    rb = sub.groupby("position_group")[b].rank(pct=True)
    return stats.spearmanr(ra, rb).statistic, len(sub)


checks = [
    ("carrier", "exploit_rate_shrunk", "pff_passing_mean"),
    ("carrier", "likely_gap_shrunk", "pff_passing_mean"),
    ("runner", "space_held_per90_poss_shrunk", "pff_receiving_per90"),
    ("runner", "receptions_per90_poss_shrunk", "pff_receiving_per90"),
    ("defender", "conceded_per90_def_shrunk", "pff_defending_mean"),
    ("defender", "conceded_per90_def_shrunk", "pff_defending_per90"),
]
rows = []
for role, ours, theirs in checks:
    rho, n = within(bench[bench[role]], ours, theirs)
    rows.append({"role": role, "ours": ours, "PFF": theirs, "rho_within_position": rho, "n": n})
print(pd.DataFrame(rows).round(2).to_string(index=False))

    role                         ours                 PFF  rho_within_position   n
 carrier          exploit_rate_shrunk    pff_passing_mean                 0.02 383
 carrier            likely_gap_shrunk    pff_passing_mean                 0.05 383
  runner space_held_per90_poss_shrunk pff_receiving_per90                -0.13 307
  runner receptions_per90_poss_shrunk pff_receiving_per90                -0.16 307
defender    conceded_per90_def_shrunk  pff_defending_mean                -0.02 307
defender    conceded_per90_def_shrunk pff_defending_per90                -0.04 307


**Better options.** PFF flags passes and carries where an analyst saw a better option. If the
decision gap measures choice, flagged actions should have larger gaps.

In [10]:
acts, flags = [], []
for f in sorted(ACTIONS_DIR.glob("pff_*.parquet")):
    a = pd.read_parquet(f)
    acts.append(a[a["status"] == "ok"])
    flags.append(better_options(f.stem.split("_", 1)[1]))
acts = pd.concat(acts).merge(pd.concat(flags), on="event_id", how="left")
acts["better_option"] = acts["better_option"].fillna(False).astype(bool)
print(f"{acts['better_option'].sum()} of {len(acts)} actions flagged "
      f"({acts['better_option'].mean():.1%})")
for col in ("likely_gap", "decision_gap"):
    auc = stats.mannwhitneyu(acts.loc[acts["better_option"], col],
                             acts.loc[~acts["better_option"], col]).statistic / (
        acts["better_option"].sum() * (~acts["better_option"]).sum())
    med = acts.groupby("better_option")[col].median()
    print(f"{col}: median {med[True]:.4f} flagged vs {med[False]:.4f} not; AUC {auc:.3f}")
miss = acts.groupby("better_option")["missed"].mean() * 100
print(f"missed rate: flagged {miss.loc[True]:.1f}%, not {miss.loc[False]:.1f}%")
named = acts[acts["better_option_player_id"].notna()]
print(f"PFF names the better player {len(named)} times; it is our likely-cell owner in "
      f"{(named['better_option_player_id'] == named['likely_owner_id']).mean():.0%}, "
      f"the raw best cell's owner in "
      f"{(named['better_option_player_id'] == named['best_owner_id']).mean():.0%}")

326 of 60285 actions flagged (0.5%)
likely_gap: median 0.0018 flagged vs 0.0019 not; AUC 0.499
decision_gap: median 0.0082 flagged vs 0.0096 not; AUC 0.473
missed rate: flagged 3.7%, not 1.9%
PFF names the better player 239 times; it is our likely-cell owner in 22%, the raw best cell's owner in 16%


## 6. V5: ratings against results

Per team-match (Spearman, all 142). Shots per possession for and against are the outcome the
ratings should track most directly; goals are noisy over one match.

In [11]:
tm["shots_pp"] = tm["n_shot"] / tm["possessions"]
tm["shots_against_pp"] = tm["n_shot_against"] / tm["opp_possessions"]
tm["box_pp"] = tm["n_box_entry"] / tm["possessions"]
tm["gd"] = tm["goals_for"] - tm["goals_against"]
rows = []
for g in ["attack_space", "attack_use", "defence_space", "defence_behind", "defence_use"]:
    row = {"grade": g}
    for out in ["shots_pp", "shots_against_pp", "box_pp", "gd"]:
        row[out] = stats.spearmanr(tm[f"grade_{g}"], tm[out]).statistic
    rows.append(row)
print(pd.DataFrame(rows).round(2).to_string(index=False))

         grade  shots_pp  shots_against_pp  box_pp    gd
  attack_space      0.20             -0.17    0.39 -0.12
    attack_use      0.34             -0.01    0.33  0.14
 defence_space      0.17             -0.20    0.30 -0.12
defence_behind      0.18             -0.24    0.30 -0.08
   defence_use      0.01             -0.34   -0.01  0.14


In [12]:
# The same at team level (World Cup, pooled over matches): stable averages.
wc = wc.assign(shots_pp=wc["n_shot"] / wc["possessions"],
               shots_against_pp=wc["n_shot_against"] / wc["opp_possessions"],
               gd_pm=(wc["n_goal"] - wc["n_goal_against"]) / wc["matches"])
for a, b in [("xspace_created", "shots_pp"), ("xspace_conceded", "shots_against_pp"),
             ("xspace_conceded_behind", "shots_against_pp"),
             ("exploit_rate_shrunk", "shots_pp"), ("opp_exploit_rate_shrunk", "shots_against_pp"),
             ("xspace_created", "gd_pm"), ("xspace_conceded", "gd_pm")]:
    r = stats.spearmanr(wc[a], wc[b])
    print(f"{a:26s} vs {b:17s} rho = {r.statistic:5.2f} (p = {r.pvalue:.3f})")

xspace_created             vs shots_pp          rho =  0.47 (p = 0.006)
xspace_conceded            vs shots_against_pp  rho =  0.38 (p = 0.034)
xspace_conceded_behind     vs shots_against_pp  rho =  0.41 (p = 0.019)
exploit_rate_shrunk        vs shots_pp          rho =  0.39 (p = 0.027)
opp_exploit_rate_shrunk    vs shots_against_pp  rho =  0.66 (p = 0.000)
xspace_created             vs gd_pm             rho =  0.13 (p = 0.469)
xspace_conceded            vs gd_pm             rho = -0.33 (p = 0.063)
